In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import matplotlib
matplotlib.style.use(snakemake.input.mpl_style)


In [ ]:
"""Barplots for AUROC and accuracy for the selected datasets/label_cols."""
scores = defaultdict(list)
for i, (label_col, dataset_macroavg_path, dataset_perlabel, dataset_name, ) in enumerate(zip(snakemake.params.label_cols, snakemake.input.datasets_macroavg, snakemake.input.datasets_perlabel, snakemake.params.datasets)):
    df=pd.read_csv(dataset_macroavg_path, index_col=0)
    df_per_class=pd.read_csv(dataset_perlabel)
    n_classes=len(df_per_class)

    scores["AUROC"].append(float(df.loc["rocauc_macroAvg"].item().replace("tensor(","").replace(")","")))
    scores["AUROC (random baseline)"].append(0.5)
    scores["Accuracy"].append(float(df.loc["accuracy_macroAvg"].item().replace("tensor(","").replace(")","")))
    scores["Accuracy (random baseline)"].append(1/n_classes)

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

plt.sca(axes[0])
plt.bar(range(len(scores["AUROC"])), scores["AUROC"],label="AUROC", color="#4d4d4dff")
for i in range (len(scores["Accuracy (random baseline)"])):
    plt.plot([i-0.4,i+0.4],[scores["AUROC (random baseline)"][i]]*2, color="#1a1a1aff", linestyle="--",label="AUROC (random baseline)")
for i, score in enumerate(scores["AUROC"]):
    plt.text(i,score+0.01,f"{round(score,2)}",ha="center", rotation=90)
#plt.xticks([])

plt.sca(axes[1])
plt.bar(range(len(scores["Accuracy"])), scores["Accuracy"],label="Accuracy", color="#4d4d4dff")
for i in range (len(scores["Accuracy (random baseline)"])):
    plt.plot([i-0.4,i+0.4],[scores["Accuracy (random baseline)"][i]]*2, color="#1a1a1aff", linestyle="--",label="Accuracy (random baseline)")
for i, score in enumerate(scores["Accuracy"]):
    plt.text(i,score+0.01,f"{round(score,2)}",ha="center", rotation=90)
#plt.xticks([])

plt.savefig(snakemake.output.macroavg_summary_plot)